# College Admission Classification Labwork

Import Modules

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

Using pandas, load "College Admission Train Data.csv"

In [ ]:
# Fonctionne depuis la racine du dépôt comme depuis le dossier notebooks
from pathlib import Path
DATA_DIR = Path("data") if Path("data").is_dir() else Path("../data")

input_data = pd.read_csv(DATA_DIR / "College Admission Train Data.csv")
print(input_data.shape)
input_data.head()

---

In this section, we will ignore the previous jury decision.


Plot the dataset using scores only :

In [ ]:
plt.scatter(input_data['Exam1'], input_data['Exam2'], s=2)
plt.xlabel('Exam1')
plt.ylabel('Exam2')
plt.axis([0, 20, 0, 20])
plt.show()

Compute the decision for each student (admitted if mean > 10) and store it in an array named decision:

In [ ]:
decision = ((input_data['Exam1'] + input_data['Exam2']) / 2 > 10).astype(int).to_numpy()
print(np.bincount(decision))  # [refusés, admis]

Plot the dataset using decision as color :

In [ ]:
plt.scatter(input_data['Exam1'], input_data['Exam2'], s=2, c=decision, cmap="prism")
plt.xlabel('Exam1')
plt.ylabel('Exam2')
plt.axis([0, 20, 0, 20])
plt.show()

In [ ]:
def plot_decision_boundary(input_data, decision, x_min=0, x_max=20, y_min=0, y_max=20, plot_step=0.05):
    xx, yy = np.meshgrid(np.arange(x_min, x_max, plot_step),
                         np.arange(y_min, y_max, plot_step))
    Z = (xx.ravel() + yy.ravel()) / 2 > 10
    Z = Z.reshape(xx.shape)
    cs = plt.pcolormesh(xx, yy, Z, cmap="Pastel1")
    plt.axis([x_min, x_max, y_min, y_max])
    plt.scatter(input_data['Exam1'], input_data['Exam2'], s=2, c=decision, cmap="prism")
    plt.show()

In [ ]:
# Frontière "moyenne > 10" avec les points colorés selon notre décision calculée
plot_decision_boundary(input_data, decision)

Plot the dicision boundaries and the data points (Apply class color from the dataset decision) :

In [ ]:
# Même frontière, points colorés selon la décision du jury
plot_decision_boundary(input_data, input_data['Decision'])

What is the algorithm decision accuracy related to the jury decision ??

In [ ]:
rule_accuracy = accuracy_score(input_data['Decision'], decision)
print(f"Accuracy de la règle 'moyenne > 10' vs jury : {rule_accuracy:.2%}")

# Linear Discriminant Analysis

Create and fit the model

In [ ]:
X_train = input_data[['Exam1', 'Exam2']].to_numpy()
y_train = input_data['Decision'].to_numpy()

lda = LinearDiscriminantAnalysis()
lda.fit(X_train, y_train)

Display the data and the decision boundaries 

In [ ]:
def plot_model_decision_boundary(model, input_data, x_min=0, x_max=20, y_min=0, y_max=20, plot_step=0.1):
    xx, yy = np.meshgrid(np.arange(x_min, x_max, plot_step),
                         np.arange(y_min, y_max, plot_step))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    cs = plt.pcolormesh(xx, yy, Z, cmap="Pastel1")
    plt.axis([x_min, x_max, y_min, y_max])
    plt.scatter(input_data['Exam1'], input_data['Exam2'], s=2, c=input_data['Decision'], cmap="prism")
    plt.show()

In [ ]:
plot_model_decision_boundary(lda, input_data)

Measure the accuracy of your model

In [ ]:
lda_accuracy = accuracy_score(y_train, lda.predict(X_train))
print(f"LDA - accuracy (train) : {lda_accuracy:.2%}")

# Quadratic Discriminant Analysis

In [ ]:
qda = QuadraticDiscriminantAnalysis()
qda.fit(X_train, y_train)

In [ ]:
plot_model_decision_boundary(qda, input_data)

In [ ]:
qda_accuracy = accuracy_score(y_train, qda.predict(X_train))
print(f"QDA - accuracy (train) : {qda_accuracy:.2%}")

---
# Decisision Tree Classifier

In [ ]:
tree = DecisionTreeClassifier(max_depth=5, random_state=0)
tree.fit(X_train, y_train)
plot_model_decision_boundary(tree, input_data)

In [ ]:
tree_accuracy = accuracy_score(y_train, tree.predict(X_train))
print(f"Decision Tree - accuracy (train) : {tree_accuracy:.2%}")

---

# Create a K-Nearest Neighbors Classifier (KNN)

In [ ]:
knn = KNeighborsClassifier(n_neighbors=15)
knn.fit(X_train, y_train)
plot_model_decision_boundary(knn, input_data)

In [ ]:
knn_accuracy = accuracy_score(y_train, knn.predict(X_train))
print(f"KNN - accuracy (train) : {knn_accuracy:.2%}")

# Random Forest

In [ ]:
forest = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=0)
forest.fit(X_train, y_train)
plot_model_decision_boundary(forest, input_data)

In [ ]:
forest_accuracy = accuracy_score(y_train, forest.predict(X_train))
print(f"Random Forest - accuracy (train) : {forest_accuracy:.2%}")

---

Load 'College Admission Test Data.csv'

In [ ]:
test_data = pd.read_csv(DATA_DIR / "College Admission Test Data.csv")
X_test = test_data[['Exam1', 'Exam2']].to_numpy()
y_test = test_data['Decision'].to_numpy()
print(test_data.shape)
test_data.head()

Compute and print the accuracy of all models on test data

In [ ]:
test_rule = ((test_data['Exam1'] + test_data['Exam2']) / 2 > 10).astype(int)
print(f"{'Règle moyenne > 10':<20}: {accuracy_score(y_test, test_rule):.2%}")

models = {'LDA': lda, 'QDA': qda, 'Decision Tree': tree, 'KNN': knn, 'Random Forest': forest}
for name, model in models.items():
    print(f"{name:<20}: {accuracy_score(y_test, model.predict(X_test)):.2%}")

**Conclusion**

- La règle simple « moyenne > 10 » reproduit la décision du jury à ~93 % : le jury ne suit donc pas exactement la moyenne.
- LDA trace une frontière linéaire et fait à peine mieux que la règle (93,8 % en test).
- Les modèles non linéaires (QDA, arbre, KNN, forêt aléatoire) captent mieux la forme courbe de la frontière du jury : ~96 % en test.
- La forêt aléatoire obtient le meilleur score en test (96,3 %), suivie de près par KNN (96,1 %).
- L'arbre de décision perd plus entre train (97,3 %) et test (95,9 %) : il sur-apprend un peu, d'où l'intérêt de limiter `max_depth`.